In [ ]:
! pip install pandas openpyxl
! pip install seaborn
! pip install streamlit seaborn matplotlib pandas

In [ ]:
import pandas as pd
import math
import unicodedata
import re

## CODIGO PARA CONVERTER O CLUSTER DE NEGOCIO, USANDO O DA AREA TECNICA, PARA A NOVA CLUSTERIZAÇÃO ##

In [3]:
# Ler a planilha da Marcelina
file_path = r"C:\Users\wnsos\OneDrive\Escritorio\Claro\Cluster_projeto_area_tecnica\cluster_marcelina.xlsx"
df = pd.read_excel(file_path, sheet_name="cluster_marcelina")

# Ver primeiras linhas
df.head()

,COD_NODE,REGIONAL,COD_IBGE,DESC_CIDADE,DESC_COD_CIDADE,CLUSTER_GEOGRAFICO,AREA,Unnamed: 7
0,JDA0040307,SP1,3556503,VARZEA PAULISTA,GRANDE SÃO PAULO,JUNDIAI,Metropolitana (Área 3),AREA VARZEA PAU
1,JDA0040308,SP1,3556503,VARZEA PAULISTA,GRANDE SÃO PAULO,JUNDIAI,Metropolitana (Área 3),AREA VARZEA PAU
2,JDA0040401,SP1,3556503,VARZEA PAULISTA,GRANDE SÃO PAULO,JUNDIAI,Metropolitana (Área 3),AREA VARZEA PAU
3,JDA0040402,SP1,3556503,VARZEA PAULISTA,GRANDE SÃO PAULO,JUNDIAI,Metropolitana (Área 3),AREA VARZEA PAU
4,JDA0040403,SP1,3556503,VARZEA PAULISTA,GRANDE SÃO PAULO,JUNDIAI,Metropolitana (Área 3),AREA VARZEA PAU


In [4]:
# Ler a planilha da Técnica
file_path = r"C:\Users\wnsos\OneDrive\Escritorio\Claro\Cluster_projeto_area_tecnica\Base_Qualinet.xlsx"
df_tec = pd.read_excel(file_path, sheet_name="Planilha1")

# Ver primeiras linhas
df_tec.head()

,PERÍODO,REGIONAL,SIGLA,SUBCLUSTER,CIDADE,AREA,NODE,GRUPO,HPs,DOMICILIO,PAYTV,PTV DIGITAL,PTV HD,VIRTUA,NETFONE,EBT PME,COD CIDADE + AREA
0,2026-02-01,TODAS,AMIO,SUBCLUSTER SÃO ROQUE,ALUMINIO,AREA ALUMINIO,3501152-SP,-,-,63,0,0,45,0,0,0,535 AREA ALUMINIO
1,2026-02-01,TODAS,AMIO,SUBCLUSTER SÃO ROQUE,ALUMINIO,AREA ALUMINIO,VLP0010101,-,22,7,0,0,1,7,3,0,535 AREA ALUMINIO
2,2026-02-01,TODAS,AMIO,SUBCLUSTER SÃO ROQUE,ALUMINIO,AREA ALUMINIO,VLP0010102,-,10,4,0,0,0,4,2,0,535 AREA ALUMINIO
3,2026-02-01,TODAS,AMIO,SUBCLUSTER SÃO ROQUE,ALUMINIO,AREA ALUMINIO,VLP0010103,-,17,7,0,0,0,7,1,0,535 AREA ALUMINIO
4,2026-02-01,TODAS,AMIO,SUBCLUSTER SÃO ROQUE,ALUMINIO,AREA ALUMINIO,VLP0010104,-,17,7,0,0,1,7,4,0,535 AREA ALUMINIO


In [5]:
# Padronizar nomes das cidades

def normalizar_texto(texto):
    if pd.isnull(texto):
        return texto
    
    # remover acentos
    texto = unicodedata.normalize('NFKD', texto)
    texto = texto.encode('ASCII', 'ignore').decode('utf-8')
    
    # remover espaços duplicados
    texto = re.sub(r'\s+', ' ', texto)
    
    # strip e upper
    texto = texto.strip().upper()
    
    return texto

In [6]:
# Aplicar a função de normalização nas colunas de cidade

df_tec['CIDADE_NORM'] = df_tec['CIDADE'].apply(normalizar_texto)
df['DESC_CIDADE_NORM'] = df['DESC_CIDADE'].apply(normalizar_texto)

In [7]:
# Validar se existe cidade no df_tec que não existe no df

cidades_tec = set(df_tec['CIDADE_NORM'].unique())
cidades_ref = set(df['DESC_CIDADE_NORM'].unique())

cidades_nao_encontradas = cidades_tec - cidades_ref

print("Cidades não encontradas:")
print(cidades_nao_encontradas)

Cidades não encontradas:
set()


In [8]:
# Verififcar se há duplicidade no COD_IBGE

df.groupby('DESC_CIDADE_NORM')['COD_IBGE'].nunique().sort_values(ascending=False).head(10)

DESC_CIDADE_NORM
ALUMINIO           1
SANTA ISABEL       1
MAUA               1
MOGI DAS CRUZES    1
MORUNGABA          1
OSASCO             1
PIRACAIA           1
POA                1
RIBEIRAO PIRES     1
SALTO              1
Name: COD_IBGE, dtype: int64

In [9]:
# Passar o COD_IBGE para a base técnica

df_ref = (
    df[['DESC_CIDADE_NORM', 'COD_IBGE']]
    .drop_duplicates(subset='DESC_CIDADE_NORM')
)

mapa_ibge = df_ref.set_index('DESC_CIDADE_NORM')['COD_IBGE']

df_tec['COD_IBGE'] = df_tec['CIDADE_NORM'].map(mapa_ibge)

In [10]:
# verificar quantos valores nulos existem na coluna COD_IBGE após o merge
df_tec['COD_IBGE'].isnull().sum()

0

In [11]:
# Padronizar NODE nas duas bases

def normalizar_node(texto):
    if pd.isnull(texto):
        return texto
    
    texto = str(texto).strip().upper()
    return texto

In [ ]:
# Padropnizar o nome das colunas, que tem colunas com espaços nos nomes

df_tec.columns = df_tec.columns.str.strip().str.upper()
df.columns = df.columns.str.strip().str.upper()

df_tec['COD_NODE_NORM'] = df_tec['NODE'].apply(normalizar_node)
df['COD_NODE_NORM'] = df['COD_NODE'].apply(normalizar_node)

In [14]:
# Conferir diferenças entre as bases

nodes_tec = set(df_tec['COD_NODE_NORM'].dropna().unique())
nodes_ref = set(df['COD_NODE_NORM'].dropna().unique())

nodes_somente_tec = nodes_tec - nodes_ref
nodes_somente_ref = nodes_ref - nodes_tec

print("Nodes apenas no df_tec:", len(nodes_somente_tec))
print("Nodes apenas no df:", len(nodes_somente_ref))

Nodes apenas no df_tec: 3259
Nodes apenas no df: 3228


In [16]:
# Criar nova variável COD_NODE + COD_IBGE

df_tec['COD_IBGE'] = df_tec['COD_IBGE'].astype('Int64')
df_tec['COD_IBGE_STR'] = df_tec['COD_IBGE'].astype(str)

df['COD_IBGE'] = df['COD_IBGE'].astype('Int64')
df['COD_IBGE_STR'] = df['COD_IBGE'].astype(str)

df_tec['CHAVE_NODE_IBGE'] = (
    df_tec['COD_NODE_NORM'] + "_" + df_tec['COD_IBGE_STR']
)

df['CHAVE_NODE_IBGE'] = (
    df['COD_NODE_NORM'] + "_" + df['COD_IBGE_STR']
)

In [17]:
# Verificar quantos valores nulos existem na nova chave

resumo = {
    "df_tec_total": len(df_tec),
    "df_tec_cod_ibge_null": df_tec['CHAVE_NODE_IBGE'].isnull().sum(),
    "df_total": len(df),
    "df_cod_node_null": df['CHAVE_NODE_IBGE'].isnull().sum()
}

resumo

{'df_tec_total': 30260,
 'df_tec_cod_ibge_null': 0,
 'df_total': 30542,
 'df_cod_node_null': 0}

In [18]:
# Validar unicidade da chave CHAVE_NODE_IBGE na df_tec

df_tec['CHAVE_NODE_IBGE'].is_unique

True

In [19]:
# Preparar para criar a coluna Area na base de negócio a partir da base técnica

mapa_area = df_tec.set_index('CHAVE_NODE_IBGE')['AREA']

df['AREA_TECNICA'] = df['CHAVE_NODE_IBGE'].map(mapa_area)

In [24]:
# Preparar para criar a coluna HP_TECNICA na base de negócio a partir da base técnica

df_tec_hp = (
    df_tec
    .dropna(subset=['CHAVE_NODE_IBGE'])
    .drop_duplicates(subset=['CHAVE_NODE_IBGE'])
    [['CHAVE_NODE_IBGE', 'HPS']]
)

df = df.merge(df_tec_hp, on='CHAVE_NODE_IBGE', how='left')
df.rename(columns={'HPS': 'HP_TECNICA'}, inplace=True)

In [25]:
# Tratar a coluna de HP antes

df['HP_TECNICA'] = (
    pd.to_numeric(df['HP_TECNICA'], errors='coerce')  
    .fillna(0)                                       
    .astype(int)                                   
)

In [26]:
df['HP_TECNICA'].dtype
df['HP_TECNICA'].isnull().sum()

0

In [ ]:
# Ler a planilha da homepass
file_path = r"C:\Users\wnsos\OneDrive\Escritorio\Claro\Cluster_projeto_area_tecnica\cluster_marcelina.xlsx"
df = pd.read_excel(file_path, sheet_name="cluster_marcelina")

# Ver primeiras linhas
df.head()

In [ ]:
# Ler a planilha da Técnica
file_path = r"C:\Users\wnsos\OneDrive\Escritorio\Claro\Cluster_projeto_area_tecnica\Base_Qualinet.xlsx"
df_tec = pd.read_excel(file_path, sheet_name="Planilha1")

# Ver primeiras linhas
df_tec.head()

In [51]:
# Criar a relação da nova clusterização com a área técnica

alto_valor = [
"AREA 01","AREA 02","AREA 04","AREA 05","AREA 10","AREA 13","AREA 17",
"AREA 19","AREA 20","AREA 21","AREA 23","AREA 28","AREA 30","AREA 41",
"AREA 46","AREA 49"
]

massivo = [
"AREA 14","AREA 15","AREA 16","AREA 34","AREA 35","AREA 37",
"AREA 39","AREA 40","AREA 48","AREA 51","AREA 53","AREA 54",
"AREA 55","AREA 56"
]

In [60]:
# Criar a função de classificação

def definir_cluster(row):

    cluster_geo = row["CLUSTER_GEOGRAFICO"]
    cidade = row["DESC_CIDADE"]
    area = row["AREA_TECNICA"]

    # EXCEÇÃO: SAO CAETANO DO SUL
    if cidade == "SAO CAETANO DO SUL":

        return pd.Series([
            "ALTO_VALOR",
            "SAO CAETANO DO SUL"
        ])

    # CASOS FORA DE SP
    if cluster_geo != "SP":

        return pd.Series([
            "METROPOLITANA",
            cidade
        ])

    # CASOS SP ALTO VALOR
    if cluster_geo == "SP" and area in alto_valor:

        return pd.Series([
            "ALTO_VALOR",
            area
        ])

    # CASOS SP MASSIVO
    if cluster_geo == "SP" and area in massivo:

        return pd.Series([
            "MASSIVO",
            area
        ])

    # fallback
    return pd.Series([
        "OUTRO",
        area
    ])

In [61]:
# Criar novas colunas

df[["CLUSTER","SUBCLUSTER"]] = df.apply(definir_cluster, axis=1)

In [62]:
# Verificar resultado

df[["CLUSTER","SUBCLUSTER"]] = df.apply(definir_cluster, axis=1)

In [63]:
# Validação

df[['DESC_CIDADE', 'AREA_TECNICA', 'CLUSTER', 'SUBCLUSTER']].sample(n=10, random_state=42)

,DESC_CIDADE,AREA_TECNICA,CLUSTER,SUBCLUSTER
18283,JARINU,AREA JARINU,METROPOLITANA,JARINU
16987,CAMPO LIMPO PAULISTA,AREA CAMPO LIMP,METROPOLITANA,CAMPO LIMPO PAULISTA
29747,MAUA,AREA 16,METROPOLITANA,MAUA
9658,SAO PAULO,AREA 28,ALTO_VALOR,AREA 28
3858,SAO ROQUE,AREA SAO ROQUE,METROPOLITANA,SAO ROQUE
11274,ITATIBA,AREA ITATIBA,METROPOLITANA,ITATIBA
6434,SAO PAULO,AREA 37,MASSIVO,AREA 37
26203,SAO BERNARDO DO CAMPO,AREA 10,METROPOLITANA,SAO BERNARDO DO CAMPO
12063,SAO PAULO,NaN,OUTRO,NaN
13019,ARUJA,AREA ARUJA,METROPOLITANA,ARUJA


In [64]:
# Validação para quando SUBCLUSTER é SAO CAETANO DO SUL

df[df['DESC_CIDADE'] == 'SAO CAETANO DO SUL'][['DESC_CIDADE', 'AREA_TECNICA', 'CLUSTER', 'SUBCLUSTER']].drop_duplicates()

,DESC_CIDADE,AREA_TECNICA,CLUSTER,SUBCLUSTER
9376,SAO CAETANO DO SUL,NaN,ALTO_VALOR,SAO CAETANO DO SUL
26557,SAO CAETANO DO SUL,AREA 12,ALTO_VALOR,SAO CAETANO DO SUL


In [56]:
df.head()

,COD_NODE,REGIONAL,COD_IBGE,DESC_CIDADE,DESC_COD_CIDADE,CLUSTER_GEOGRAFICO,AREA,UNNAMED: 7,DESC_CIDADE_NORM,COD_NODE_NORM,COD_IBGE_STR,CHAVE_NODE_IBGE,AREA_TECNICA,HP_TECNICA,CLUSTER,SUBCLUSTER
0,JDA0040307,SP1,3556503,VARZEA PAULISTA,GRANDE SÃO PAULO,JUNDIAI,Metropolitana (Área 3),AREA VARZEA PAU,VARZEA PAULISTA,JDA0040307,3556503,JDA0040307_3556503,AREA VARZEA PAU,13,METROPOLITANA,VARZEA PAULISTA
1,JDA0040308,SP1,3556503,VARZEA PAULISTA,GRANDE SÃO PAULO,JUNDIAI,Metropolitana (Área 3),AREA VARZEA PAU,VARZEA PAULISTA,JDA0040308,3556503,JDA0040308_3556503,AREA VARZEA PAU,19,METROPOLITANA,VARZEA PAULISTA
2,JDA0040401,SP1,3556503,VARZEA PAULISTA,GRANDE SÃO PAULO,JUNDIAI,Metropolitana (Área 3),AREA VARZEA PAU,VARZEA PAULISTA,JDA0040401,3556503,JDA0040401_3556503,AREA VARZEA PAU,18,METROPOLITANA,VARZEA PAULISTA
3,JDA0040402,SP1,3556503,VARZEA PAULISTA,GRANDE SÃO PAULO,JUNDIAI,Metropolitana (Área 3),AREA VARZEA PAU,VARZEA PAULISTA,JDA0040402,3556503,JDA0040402_3556503,AREA VARZEA PAU,17,METROPOLITANA,VARZEA PAULISTA
4,JDA0040403,SP1,3556503,VARZEA PAULISTA,GRANDE SÃO PAULO,JUNDIAI,Metropolitana (Área 3),AREA VARZEA PAU,VARZEA PAULISTA,JDA0040403,3556503,JDA0040403_3556503,AREA VARZEA PAU,19,METROPOLITANA,VARZEA PAULISTA
